In [1]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torch.optim as optim
import re

from tqdm import trange, tqdm

ciph = np.load('cipher.npy')

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()


In [2]:
GPU_indx = 0
device = torch.device(GPU_indx if torch.cuda.is_available() else 'cpu')
print(device)

cuda:0


In [3]:
train = ciph[:, 0:1616, :]
valid = ciph[:, 1616:1818, :]
test = ciph[:, 1818:2020, :]
print(train.shape)

(27, 1616, 50)


In [4]:
def nums_to_chars(arr):
    lookup = "abcdefghijklmnopqrstuvwxyz "
    return np.vectorize(lambda x: lookup[int(x)])(arr)

In [5]:
y_train = torch.tensor(np.repeat(np.arange(27), train.shape[1]))
x_train = torch.tensor(train.reshape((-1,50)))

y_valid = torch.tensor(np.repeat(np.arange(27), valid.shape[1]))
x_valid = torch.tensor(valid.reshape((-1,50)))

y_test = torch.tensor(np.repeat(np.arange(27), test.shape[1]))
x_test = torch.tensor(test.reshape((-1,50)))

print(ciph.min())
print(ciph.max())

0
26


In [6]:
class ShakeText(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

In [7]:
batch_size = 64
train_loader = DataLoader(ShakeText(x_train, y_train), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(ShakeText(x_test, y_test), batch_size=batch_size, shuffle=False)
valid_loader = DataLoader(ShakeText(x_valid, y_valid), batch_size=batch_size, shuffle=False)

In [8]:
x, y = next(iter(train_loader))

print(x.shape)
print(y.shape)
print(x.dtype)
print(y.dtype)

torch.Size([64, 50])
torch.Size([64])
torch.uint8
torch.int64


In [18]:
class NN(nn.Module):
    def __init__(self, num_classes):
        super(NN, self).__init__()

        self.embedding = nn.Embedding(27, 16)
        self.fc1 = nn.Linear(16*50, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, num_classes)

    def forward(self, x):
        
        x = x.long()
        x = self.embedding(x)
        x = x.flatten(1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.fc4(x)
        return x 

In [21]:
model = NN(27).to(device)
criterion = nn.CrossEntropyLoss()
lr = 1e-3
optimizer = optim.Adam(model.parameters(), lr)
n_epochs = 20

print(model)

NN(
  (embedding): Embedding(27, 16)
  (fc1): Linear(in_features=800, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=128, bias=True)
  (fc4): Linear(in_features=128, out_features=27, bias=True)
)


In [11]:
def train_epoch(model, train_loader, criterion, optimizer, loss_logger):
    for batch_idx, (data, target) in enumerate(tqdm(train_loader, desc="Training", leave=False)):
        
        outputs = model(data.to(device))
        loss = criterion(outputs, target.to(device))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loss_logger.append(loss.item())

    return model, optimizer, loss_logger

In [12]:
def test_model(model, test_loader, criterion, loss_logger):
    with torch.no_grad():
        correct_predictions = 0
        total_predictions = 0
        
        for batch_idx, (data, target) in enumerate(tqdm(test_loader, desc="Testing", leave=False)):   
            
            outputs = model(data.to(device))        
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == target.to(device)).sum().item()
            total_predictions += target.shape[0]
            loss = criterion(outputs, target.to(device))
            loss_logger.append(loss.item())
            
        acc = (correct_predictions/total_predictions) * 100.0
        return loss_logger, acc

In [13]:
train_loss = []
test_loss  = []
test_acc   = []

In [40]:
for i in trange(n_epochs, desc="Epoch", leave=False):
    model, optimizer, train_loss = train_epoch(model, train_loader, criterion, optimizer, train_loss)
    
    test_loss, acc = test_model(model, test_loader, criterion, test_loss)
    test_acc.append(acc)

print("Final Accuracy: %.2f%%" % acc)

Final Accuracy: 98.79%


In [47]:
torch.save(model.state_dict(), "model.pth")